In [2]:
import nltk

import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# from textblob import TextBlob
from pathlib import Path
from nltk.tokenize import sent_tokenize
from nltk.tokenize import word_tokenize
from tqdm import tqdm
import contractions
from nltk import pos_tag
from nltk.corpus import stopwords
import matplotlib.ticker as ticker
import re
from sklearn.model_selection import train_test_split
import torch.nn.functional as F
import torch
from nltk.corpus import stopwords  
from gensim.models import Word2Vec

import torch
import torch.nn as nn
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report
from transformers import AutoModel, AutoTokenizer

c:\Users\alvar\OneDrive\Escritorio\clean_nlp_env\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def select_text(entry_1, entry_2, entry_3):
    text = entry_2 if entry_3 == 1 else entry_1
    return text

def get_text(df, entry_1, entry_2, entry_3):

    df['final_text'] = df.progress_apply(
        lambda row: select_text(row[entry_1], row[entry_2], row[entry_3]),
        axis=1
    )

    return df

def preprocess_text(entry_1):
    # Choose text source based on count_climate_change
    text = entry_1

    custom_stopwords = {'mr', 'there', 'that', 'this', 'they', 'them', 'those', 'themselves'} | set(stop_words)

    tokens = text.split()
    
    return [token for token in tokens
        if (token.lower() not in custom_stopwords)  # Case-insensitive check
        and (len(token) > 3)]

def preprocess_df(df, column_1):
    tqdm.pandas()
    
    # Apply preprocessing and store results in 'tokens' column
    df['tokens'] = df.progress_apply(
        lambda row: preprocess_text(row[column_1]),
        axis=1
    )
    
    return df

In [3]:
class TokenDataset(Dataset):
    def __init__(self, token_lists, labels, w2v_model, pooling='aver', window_size=10):
        """
        Args:
            token_lists: List of tokenized texts
            labels: List of corresponding labels
            w2v_model: Trained Word2Vec model
            pooling: Pooling method ('aver', 'max', 'hier', 'concat')
            window_size: Size of window for hierarchical pooling
        """
        self.tokens = token_lists
        self.labels = labels
        self.wv = w2v_model.wv
        self.pooling = pooling
        self.window_size = window_size
        self.embedding_dim = self.wv.vector_size
        
    def __len__(self):
        return len(self.labels)
    
    def __getitem__(self, idx):
        tokens = self.tokens[idx]
        label = self.labels[idx]
        
        # Convert tokens to embeddings
        vectors = [self.wv[word] for word in tokens if word in self.wv]
        
        if not vectors:  # Handle empty cases
            vectors = [np.zeros(self.embedding_dim)]
            
        vectors = np.array(vectors)
        
        # Apply selected pooling method
        if self.pooling == 'aver':
            embedding = vectors.mean(axis=0)
        elif self.pooling == 'max':
            embedding = vectors.max(axis=0)
        elif self.pooling == 'hier':
            if len(vectors) < self.window_size:
                embedding = vectors.mean(axis=0)
            else:
                windows = [vectors[i:i+self.window_size] 
                           for i in range(len(vectors) - self.window_size + 1)]
                local_means = [np.mean(w, axis=0) for w in windows]
                embedding = np.max(local_means, axis=0)
        elif self.pooling == 'concat':
            embedding = np.concatenate([
                vectors.mean(axis=0),
                vectors.max(axis=0)
            ])
        else:
            raise ValueError(f"Unknown pooling method: {self.pooling}")
        
        # Convert to tensors
        return (
            torch.FloatTensor(embedding),
            torch.LongTensor([label])  # Keep as 1D tensor for compatibility
        )
    
    @staticmethod
    def collate_fn(batch):
        """Custom collate function to handle variable-length sequences"""
        embeddings = torch.stack([item[0] for item in batch])
        labels = torch.cat([item[1] for item in batch])  # Concatenate 1D tensors
        return embeddings, labels


In [4]:
class SWEMClassifier(nn.Module):
    def __init__(self, embedding_model, num_classes=4):
        super().__init__()
        self.wv = embedding_model.wv
        self.embedding_dim = self.wv.vector_size

        input_dim = self.embedding_dim

        # Define the classification head
        self.classifier = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        """Forward pass - expects already pooled embeddings"""
        return self.classifier(x)

In [5]:
import pickle

def save_training_history(history, filename="training_history.pkl"):
    with open(filename, "wb") as f:
        pickle.dump(history, f)

In [6]:
def plot_training_history(history, method = None, lr = None):
    plt.figure(figsize=(12, 5))
    
    # Loss plot
    plt.subplot(1, 2, 1)
    plt.plot(history['train_loss'], label='Train Loss')
    plt.plot(history['val_loss'], label='Validation Loss')
    plt.title('Training and Validation Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plot_path = Path(r"C:\Users\alvar\OneDrive\Escritorio\BDS\Block_5\NLP\word2vec") / f"loss_word2vec_{method}_&_{lr}.png"
    plt.savefig(plot_path)
    plt.legend()
    
    # Accuracy plot
    plt.subplot(1, 2, 2)
    plt.plot(history['train_acc'], label='Train Accuracy')
    plt.plot(history['val_acc'], label='Validation Accuracy')
    plt.title('Training and Validation Accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.legend()
    plot_path = Path(r"C:\Users\alvar\OneDrive\Escritorio\BDS\Block_5\NLP\word2vec") / f"acc_word2vec_{method}_&_{lr}.png"
    plt.savefig(plot_path)
    plt.tight_layout()
    #plt.show()
    
    # F1-score plot
    plt.figure(figsize=(6, 4))
    plt.plot(history['train_f1'], label='Train F1')
    plt.plot(history['val_f1'], label='Validation F1')
    plt.title('Training and Validation F1 Score')
    plt.xlabel('Epoch')
    plt.ylabel('F1 Score')
    plt.legend()
    plot_path = Path(r"C:\Users\alvar\OneDrive\Escritorio\BDS\Block_5\NLP\word2vec")  / f"f1_word2vec_{method}_&_{lr}.png"
    plt.savefig(plot_path)
    #plt.show()

def plot_confusion_matrix(cm, classes, method = None, lr = None):
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=classes, yticklabels=classes)
    plt.title('Confusion Matrix')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.tight_layout()
    plot_path = Path(r"C:\Users\alvar\OneDrive\Escritorio\BDS\Block_5\NLP\word2vec") / f"final_model_confusion_matrix_{method}_&_{lr}.png"
    plt.savefig(plot_path)
    #plt.show()

    
def evaluate_model(model, data_loader, criterion, device, num_classes):
    model.eval()
    metrics = {
        'loss': 0, 'correct': 0, 'total': 0,
        'all_preds': [], 'all_labels': []
    }
    
    with torch.no_grad():
        for embeddings, labels in data_loader:
            embeddings, labels = embeddings.to(device), labels.to(device)
            outputs = model(embeddings)
            loss = criterion(outputs, labels.squeeze())
            
            metrics['loss'] += loss.item() * labels.size(0)
            _, predicted = torch.max(outputs.data, 1)
            metrics['correct'] += (predicted == labels.squeeze()).sum().item()
            metrics['total'] += labels.size(0)
            metrics['all_preds'].extend(predicted.cpu().numpy())
            metrics['all_labels'].extend(labels.squeeze().cpu().numpy())
    
    metrics['loss'] /= metrics['total']
    metrics['acc'] = metrics['correct'] / metrics['total']
    
    # Calculate F1 score and confusion matrix
    report = classification_report(
        metrics['all_labels'],
        metrics['all_preds'],
        output_dict=True,
        zero_division=0
    )
    metrics['f1'] = report['macro avg']['f1-score']
    metrics['conf_matrix'] = confusion_matrix(
        metrics['all_labels'],
        metrics['all_preds'],
        labels=range(num_classes)
    )
    
    return metrics

In [ ]:
def train_model(model, train_loader, val_loader, test_loader=None, epochs=10, lr_1=None, early_stopping_patience=15, num_classes=4, method_1=None):
    # Add this near the top of your script
    class_names = ["Low", "Lower-Middle", "Upper-Middle", "High"]


    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    
    optimizer = torch.optim.Adam(model.parameters(), lr=lr_1)
    criterion = nn.CrossEntropyLoss()
    
    best_val_loss = float('inf')
    patience_counter = 0
    history = {
        'train_loss': [], 'train_acc': [], 
        'val_loss': [], 'val_acc': [],
        'train_f1': [], 'val_f1': [],
        'confusion_matrices': []
    }

    for epoch in range(epochs):
        # Training phase
        model.train()
        train_metrics = {
            'loss': 0, 'correct': 0, 'total': 0,
            'all_preds': [], 'all_labels': []
        }

        for embeddings, labels in train_loader:
            embeddings, labels = embeddings.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(embeddings)
            loss = criterion(outputs, labels.squeeze())
            loss.backward()
            optimizer.step()

            train_metrics['loss'] += loss.item() * labels.size(0)
            _, predicted = torch.max(outputs.data, 1)
            train_metrics['correct'] += (predicted == labels.squeeze()).sum().item()
            train_metrics['total'] += labels.size(0)
            train_metrics['all_preds'].extend(predicted.cpu().numpy())
            train_metrics['all_labels'].extend(labels.squeeze().cpu().numpy())

        # Calculate training metrics
        train_loss = train_metrics['loss'] / train_metrics['total']
        train_acc = train_metrics['correct'] / train_metrics['total']
        train_report = classification_report(
            train_metrics['all_labels'],
            train_metrics['all_preds'],
            output_dict=True,
            zero_division=0
        )
        train_f1 = train_report['macro avg']['f1-score']
        
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['train_f1'].append(train_f1)

        # Validation phase
        val_metrics = evaluate_model(model, val_loader, criterion, device, num_classes)
        history['val_loss'].append(val_metrics['loss'])
        history['val_acc'].append(val_metrics['acc'])
        history['val_f1'].append(val_metrics['f1'])
        history['confusion_matrices'].append(val_metrics['conf_matrix'])

        # Early stopping check
        if val_metrics['loss'] < best_val_loss:
            best_val_loss = val_metrics['loss']
            patience_counter = 0
            torch.save(model.state_dict(), 'best_model.pth')
        else:
            patience_counter += 1
            if patience_counter >= early_stopping_patience:
                print(f"Early stopping at epoch {epoch+1}")
                break

        print(f"Epoch {epoch+1}/{epochs}")
        print(f"Train Loss: {train_loss:.4f} | Acc: {train_acc:.4f} | F1: {train_f1:.4f}")
        print(f"Val   Loss: {val_metrics['loss']:.4f} | Acc: {val_metrics['acc']:.4f} | F1: {val_metrics['f1']:.4f}")
        print("-" * 50)

    model.load_state_dict(torch.load('best_model.pth'))

    plot_training_history(history, method=method_1, lr=lr_1)

    if test_loader:
        test_metrics = evaluate_model(model, test_loader, criterion, device, num_classes)
        print("\nTest Set Performance:")
        print(f"Loss: {test_metrics['loss']:.4f} | Acc: {test_metrics['acc']:.4f} | F1: {test_metrics['f1']:.4f}")
        
        print("\nClassification Report:")
        print(classification_report(
            test_metrics['all_labels'],
            test_metrics['all_preds'],
            target_names=class_names
        ))

        plot_confusion_matrix(test_metrics['conf_matrix'], class_names, method = method_1, lr=lr_1)
        history['test_metrics'] = test_metrics

    return model, history


In [8]:
def extract_context_or_full(tokens, count, keyword="climate_change", window=10):
    if count == 1:
        for i, word in enumerate(tokens):
            if word == keyword:
                start = max(0, i - window)
                end = min(len(tokens), i + window + 1)
                return tokens[start:end]  # include keyword
        return []  # fallback if keyword somehow not found
    else:
        return tokens  # use full speech

In [9]:
stop_words = set(stopwords.words('english'))

# Step 1: Load the speeches

In [10]:
base_path_alvaro = Path(r"C:\Users\alvar\OneDrive\Escritorio\BDS\Block_5\NLP\Project")

base_path = base_path_alvaro # change according to user

In [11]:
#df.to_csv(base_path_alvaro / "DF_word_embeddings.csv", index=False, encoding='utf-8')
df_wo_tokens = pd.read_csv(base_path_alvaro / "DF_word_embeddings.csv")

df_w_tokens = preprocess_df(df_wo_tokens.copy(), column_1='final_text')

df_w_tokens

100%|██████████| 2946/2946 [00:00<00:00, 3693.96it/s]


,Year,ISO-Code,Income Level,speeches_for_word2vec,climate_sentences_extended,count_climate_change,final_text,tokens
0,1990,BGD,1,warm felicitations are due you on your wellde...,the conference must produce results that will ...,1,the conference must produce results that will ...,"[conference, must, produce, results, assist, c..."
1,1990,CHN,1,i should like to begin by warmly congratulatin...,looking forward into the 1990s we see a world ...,1,looking forward into the 1990s we see a world ...,"[looking, forward, 1990s, world, faced, challe..."
2,1990,COD,1,the fortyfifth session of the united_nation g...,an increase in the planets average temperature...,1,an increase in the planets average temperature...,"[increase, planets, average, temperature, lead..."
3,1990,DNK,4,i congratulate you sir on your election as pre...,at the same time we must not lose sight of oth...,4,i congratulate you sir on your election as pre...,"[congratulate, election, president, general, a..."
4,1990,ISL,4,to congratulate you on your election to our h...,international treaties in specific fields of t...,1,international treaties in specific fields of t...,"[international, treaties, specific, fields, en..."
...,...,...,...,...,...,...,...,...
2941,2024,WSM,2,excellencies i extend my congratulations to hi...,please be assured of samoas support in the suc...,7,excellencies i extend my congratulations to hi...,"[excellencies, extend, congratulations, excell..."
2942,2024,YEM,1,ladies and gentlemen it is a happy coincidence...,for this reason the republic of yemen renews i...,1,for this reason the republic of yemen renews i...,"[reason, republic, yemen, renews, call, intern..."
2943,2024,ZAF,3,president of the 79th session of the un genera...,extreme_weather such as flooding fires and dro...,2,president of the 79th session of the un genera...,"[president, 79th, session, general, assembly, ..."
2944,2024,ZMB,2,ladies and gentlemen i congratulate you your e...,furthermore zambia recognises the efforts of h...,2,ladies and gentlemen i congratulate you your e...,"[ladies, gentlemen, congratulate, excellency, ..."


In [12]:
df_w_tokens.iloc[0]['final_text']

'the conference must produce results that will assist countries particularly in the developing world to meet their obligations we hope the proposed conventions on climate_change and on the protection of biodiversity will soon be ready for signature a mainstay of the consolidation of global peace development and security is the effort to strengthen the rule of international law'

# 1. word2vec

In [13]:
df = df_w_tokens.drop(['ISO-Code', 'speeches_for_word2vec', 'climate_sentences_extended', 'Year', 'final_text'], axis = 1)

df['Income Level Encoded'] = df['Income Level'] - 1

df = df.drop(['Income Level'], axis = 1)

X_train, X_test, y_train, y_test = train_test_split(df[['tokens', 'count_climate_change']], df['Income Level Encoded'], test_size=0.2, random_state=42)

## Train model

In [15]:
w2v_model = Word2Vec(
            sentences=X_train['tokens'].tolist(),
            vector_size=150,
            window=10,
            min_count=1,  # Increased from 1 for better quality
            sample=1e-5,
            workers=4,
            sg=1,
            seed=42,
            epochs=10
        )


## Extract context

In [ ]:
tqdm.pandas()

# First split: 80% train+val, 20% test (unchanged)
X_trainval, X_test, y_trainval, y_test = train_test_split(
    df[['tokens', 'count_climate_change']], 
    df['Income Level Encoded'], 
    test_size=0.125, 
    random_state=42,
    stratify=df['Income Level Encoded']
)

# Second split: 80% train, 20% validation (of the remaining 80%)
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval,
    y_trainval,
    test_size=0.14,  # 0.14 * 0.875 = 0.1225 of original
    random_state=42,
    stratify=y_trainval
)

 # Apply your context extraction
X_train_context = X_train.progress_apply(lambda row: extract_context_or_full(row['tokens'], row['count_climate_change']), axis=1)
X_val_context = X_val.progress_apply(lambda row: extract_context_or_full(row['tokens'], row['count_climate_change']), axis=1)
X_test_context = X_test.progress_apply(lambda row: extract_context_or_full(row['tokens'], row['count_climate_change']), axis=1)

100%|██████████| 590/590 [00:00<00:00, 97124.67it/s]


In [17]:
X_train_context

1701    [president, general, assembly, behalf, governm...
1606    [goals, national, development, plans, also, em...
931     [human, development, indicators, like, inhabit...
509     [allow, outset, convey, warm, congratulations,...
2460    [begin, joining, previous, speakers, applaudin...
                              ...                        
2145    [behalf, excellency, desire, delano, bouterse,...
33      [environment, development, janeiro, canadas, p...
1677    [line, provisions, constitution, responsibilit...
456     [stand, assembly, first, time, president, fede...
2414    [namibia, joins, member, states, congratulatin...
Length: 1767, dtype: object

## Embedd accordignly - Run it Once

In [ ]:
methods = ['aver', 'max', 'hier']
learning_rates = [1e-3, 3e-4, 2e-4, 1e-5]

results = []

for method in methods:
    print('--------------------------------------', method.upper(), '--------------------------------')

    # Create datasets/loaders per pooling method
    train_dataset = TokenDataset(X_train_context.tolist(), y_train.tolist(), w2v_model, pooling=method)
    val_dataset = TokenDataset(X_val_context.tolist(), y_val.tolist(), w2v_model, pooling=method)
    test_dataset = TokenDataset(X_test_context.tolist(), y_test.tolist(), w2v_model, pooling=method)

    # Re-initialize model for each run
    model = SWEMClassifier(w2v_model, num_classes=4)

        # Loaders can be reused if batch size fixed
    train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=8)
    test_loader = DataLoader(test_dataset, batch_size=8)
    
    for lr in learning_rates:
        print(f"\n>> Training with pooling: {method}, lr: {lr}")
        
        # Re-initialize model for each run
        model = SWEMClassifier(w2v_model, num_classes=4)

        # Loaders can be reused if batch size fixed
        train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
        val_loader = DataLoader(val_dataset, batch_size=8)
        test_loader = DataLoader(test_dataset, batch_size=8)

        # Train
        model, history = train_model(
            model, train_loader, val_loader,
            test_loader=test_loader,
            epochs=100,
            lr_1=lr,
            early_stopping_patience=15,
            num_classes=4,
            method_1=method
        )

        # Save
        save_training_history(history, filename=f"history_{method}_lr{lr}.pkl")

        # Evaluate (optional, if not already in history)
        test_f1 = history['test_metrics']['f1']
        val_f1 = history['val_f1'][-1]  # Last recorded val F1

        results.append((method, lr, val_f1, test_f1))

""" # Store all results
results_df = pd.DataFrame(results, columns=["Pooling", "LearningRate", "ValF1", "TestF1"])
results_df = results_df.sort_values("ValF1", ascending=False)

# Save to CSV
results_df.to_csv("C:/Users/alvar/OneDrive/Escritorio/BDS/Block_5/NLP/word2vec/results_grid_search.csv", index=False)

print("\nTop Results:")
print(results_df.head()) """


-------------------------------------- AVER --------------------------------

>> Training with pooling: aver, lr: 0.001
Epoch 1/100
Train Loss: 1.3632 | Acc: 0.3033 | F1: 0.2368
Val   Loss: 1.3644 | Acc: 0.2954 | F1: 0.1140
--------------------------------------------------


KeyboardInterrupt: 

In [3]:
results_df = pd.read_csv("C:/Users/alvar/OneDrive/Escritorio/BDS/Block_5/NLP/word2vec/results_grid_search.csv")
results_df.sort_values("TestF1", ascending=False)

,Pooling,LearningRate,ValF1,TestF1
0,aver,0.00100,0.523591,0.592883
3,aver,0.00030,0.486953,0.587237
7,aver,0.00020,0.470883,0.584732
2,hier,0.00020,0.492821,0.539587
8,hier,0.00030,0.467499,0.539329
1,hier,0.00100,0.503877,0.538154
5,max,0.00020,0.475873,0.501997
6,max,0.00030,0.475424,0.501904
4,max,0.00100,0.479108,0.480160
9,max,0.00001,0.296090,0.298270
